In [29]:
"""
train_qr_diffusion.py
 
Lightweight conditional diffusion restoration of motion-blurred QR crops.
UNet + noise schedulers come from HuggingFace `diffusers` (standard
infrastructure). Project-specific parts kept custom:
  (1) SAM3-box based frame<->reference pairing with ECC/ORB homography alignment
      (pairs that fail an alignment-quality check are dropped, not trained on),
  (2) patch-based L1 + SSIM (+ optional patch-PSNR) loss on the x0 estimate,
  (3) decode-success evaluation against the GT string decoded from the
      original full-resolution reference photo, stratified by blur score,
  (4) synthetic degradation (motion/defocus/illumination/noise) for data scale,
  (5) optional one-step distilled student for fast inference.
  
"""

'\ntrain_qr_diffusion.py\n \nLightweight conditional diffusion restoration of motion-blurred QR crops.\nUNet + noise schedulers come from HuggingFace `diffusers` (standard\ninfrastructure). Project-specific parts kept custom:\n  (1) SAM3-box based frame<->reference pairing with ECC/ORB homography alignment\n      (pairs that fail an alignment-quality check are dropped, not trained on),\n  (2) patch-based L1 + SSIM (+ optional patch-PSNR) loss on the x0 estimate,\n  (3) decode-success evaluation against the GT string decoded from the\n      original full-resolution reference photo, stratified by blur score,\n  (4) synthetic degradation (motion/defocus/illumination/noise) for data scale,\n  (5) optional one-step distilled student for fast inference.\n  \n'

In [30]:
import json
import math
import os
import random
import shutil
import time
from pathlib import Path
import copy
import re

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import DDIMScheduler, DDPMScheduler, UNet2DModel
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download, snapshot_download
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

In [31]:
REPO_ID = "RakshitAi/IITB-BuggyQR"
LOCAL_DIR = "/kaggle/working/buggyqr_data"
CACHE_DIR = "/kaggle/working/qr_pairs_cache"
CKPT_DIR = "/kaggle/working/checkpoints"
 
IMG_SIZE = 256
PATCH_SIZE = 16
TIMESTEPS = 100
SAMPLE_STEPS = 10
BATCH_SIZE = 16
EPOCHS = 100
LR = 2e-4
LAMBDA_PATCH = 1.0        # weight of the patch loss relative to the v-prediction MSE
PATCH_PSNR_WEIGHT = 0.0   # >0 adds a patch-PSNR term (ablation switch)
RANDOM_PATCH_OFFSET = True
TEST_FRACTION = 0.15
TEST_GAP = 5              # frames dropped between train and test block (leakage guard)
ALIGN_NCC_THRESH = 0.5    # min alignment quality to keep a (frame, reference) pair
SYNTHETIC_PROB = 0.5      # chance a training sample's input is synthetically degraded
EMA_DECAY = 0.995
GRAD_CLIP = 1.0
SAVE_EVERY = 5
DISTILL_EPOCHS = 30
SEED = 42
 
RESUME_FROM_HUB = True
PUSH_CHECKPOINTS = True
CHECKPOINT_REPO_ID = "RakshitAi/IITB-BuggyQR-diffusion"  
 
device = "cuda" if torch.cuda.is_available() else "cpu"
 
 
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [32]:
# 0. Token
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("No HF_TOKEN found (Kaggle secret, .env, or environment variable).")
    return token

In [33]:
# 1. Build (degraded frame, clean reference) pairs from the SAM3 annotations
def crop_square(image: Image.Image, box, pad_frac=0.15):
    """Square crop around `box` (xyxy) with padding. The window is SHIFTED to
    stay inside the image rather than clipped, so the crop is always square
    (clip + square resize would stretch the QR and corrupt the pairing)."""
    W, H = image.size
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = min(max(x2 - x1, y2 - y1) * (1 + pad_frac), W, H)
    left = min(max(cx - side / 2, 0), W - side)
    top = min(max(cy - side / 2, 0), H - side)
    s, l, t = int(side), int(left), int(top)
    return image.crop((l, t, l + s, t + s))

def _bandpass(gray_u8):
    """Illumination- and blur-tolerant representation used for alignment/scoring."""
    g = gray_u8.astype(np.float32)
    return cv2.GaussianBlur(g, (0, 0), 1.5) - cv2.GaussianBlur(g, (0, 0), 12)
 
 
def _ncc(a, b):
    a = a - a.mean()
    b = b - b.mean()
    denom = float(np.sqrt((a * a).sum() * (b * b).sum()))
    return float((a * b).sum() / denom) if denom > 1e-6 else 0.0
 
 
def _orb_homography(frame_gray, ref_gray, min_matches=10):
    """Homography mapping REF coords -> FRAME coords from ORB matches, or None."""
    orb = cv2.ORB_create(2000)
    k1, d1 = orb.detectAndCompute(frame_gray, None)
    k2, d2 = orb.detectAndCompute(ref_gray, None)
    if d1 is None or d2 is None or len(k1) < min_matches or len(k2) < min_matches:
        return None
    matches = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d2, d1, k=2)
    good = [p[0] for p in matches if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < min_matches:
        return None
    src = np.float32([k2[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    if H is None or mask is None or int(mask.sum()) < min_matches:
        return None
    return H
 
 
def _ecc_refine(frame_bp, ref_bp, init_ref_to_frame):
    """ECC refinement. Returns W mapping FRAME coords -> REF coords, which is
    what cv2.warpPerspective(..., flags=WARP_INVERSE_MAP) needs to warp the
    reference onto the frame."""
    try:
        W = np.linalg.inv(init_ref_to_frame).astype(np.float32)
        W = W / W[2, 2]
        if not np.all(np.isfinite(W)):
            raise np.linalg.LinAlgError
    except np.linalg.LinAlgError:
        W = np.eye(3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5)
    try:
        _, W_ref = cv2.findTransformECC(frame_bp, ref_bp, W.copy(), cv2.MOTION_HOMOGRAPHY, criteria, None, 5)
        if np.all(np.isfinite(W_ref)):
            W = W_ref
    except cv2.error:
        pass  # did not converge: keep the initial estimate
    return W

def align_reference_to_frame(frame_bgr, ref_bgr, ncc_thresh=ALIGN_NCC_THRESH):
    """Warps the clean reference onto the degraded frame's geometry.
    Tries ORB-initialised and identity-initialised ECC, keeps the better one.
    Returns (warped_reference_or_None, alignment_score); None if the best
    score is below `ncc_thresh` (pair should not be used for training/eval)."""
    h, w = frame_bgr.shape[:2]
    frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    ref_gray = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2GRAY)
    frame_bp, ref_bp = _bandpass(frame_gray), _bandpass(ref_gray)
 
    inits = [np.eye(3, dtype=np.float64)]
    H0 = _orb_homography(frame_gray, ref_gray)
    if H0 is not None:
        inits.insert(0, H0)
 
    best_img, best_score = None, -1.0
    for init in inits:
        W = _ecc_refine(frame_bp, ref_bp, init)
        warped = cv2.warpPerspective(ref_bgr, W, (w, h), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                                     borderMode=cv2.BORDER_REPLICATE)
        score = _ncc(frame_bp, _bandpass(cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)))
        if score > best_score:
            best_img, best_score = warped, score
    return (best_img if best_score >= ncc_thresh else None), best_score

def _pick_box(boxes):
    """Highest-score SAM3 box that is roughly square (QR codes are)."""
    ok = []
    for b in boxes:
        x1, y1, x2, y2 = b["box_xyxy"]
        w, h = x2 - x1, y2 - y1
        if w > 8 and h > 8 and 0.5 <= w / h <= 2.0:
            ok.append(b)
    return max(ok, key=lambda b: b["score"])["box_xyxy"] if ok else None


def build_pairs(local_dir):
    """Pairs each frame with its reference photo by qr_id (the tagging check
    that stops a frame being paired with the wrong QR's ground truth)."""
    ann_path = os.path.join(local_dir, "annotations", "all_boxes.jsonl")
    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"{ann_path} not found -- run annotate_and_push.py first.")
 
    records = {}
    with open(ann_path) as f:
        for line in f:
            r = json.loads(line)
            records[r["file_name"]] = r
 
    ref_by_qrid = {}
    for path, r in records.items():
        if r["source"] == "reference":
            box = _pick_box(r["boxes"])
            if box is not None:
                ref_by_qrid[r["qr_id"]] = (path, box)
 
    pairs = []
    for path, r in records.items():
        if r["source"] != "frame" or r["qr_id"] not in ref_by_qrid:
            continue
        box = _pick_box(r["boxes"])
        if box is None:
            continue
        m = re.search(r"_f(\d+)\.jpg$", path)
        ref_path, ref_box = ref_by_qrid[r["qr_id"]]
        pairs.append({
            "frame_path": os.path.join(local_dir, path), "frame_box": box,
            "frame_idx": int(m.group(1)) if m else 0,
            "ref_path": os.path.join(local_dir, ref_path), "ref_box": ref_box,
            "qr_id": r["qr_id"], "session_id": r["session_id"],
        })
    return pairs

def decode_reference_string(ref_img, ref_box):
    """GT string = decode of the ORIGINAL full-resolution reference crop
    (not the downscaled training target, which may be too small to decode)."""
    crop_bgr = cv2.cvtColor(np.array(crop_square(ref_img, ref_box)), cv2.COLOR_RGB2BGR)
    for size in (None, 1024, 768, 512, 384, 256):
        im = crop_bgr if size is None else cv2.resize(crop_bgr, (size, size), interpolation=cv2.INTER_AREA)
        s = decode_qr(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        if s:
            return s
    return None


def prepare_training_pairs(local_dir, cache_dir, img_size, test_fraction=TEST_FRACTION, gap=TEST_GAP):
    manifest_path = os.path.join(cache_dir, "manifest.json")
    if os.path.exists(manifest_path):
        print(f"Found existing cache -> {manifest_path} (delete the whole cache dir to rebuild)")
        return json.load(open(manifest_path))
 
    os.makedirs(cache_dir, exist_ok=True)
    pairs = build_pairs(local_dir)
    print(f"{len(pairs)} frame/reference pairs available (matched by qr_id)")
 
    ref_cache = {}   # qr_id -> (reference crop resized to img_size, GT string)
    entries, scores, n_unaligned, n_failed = [], [], 0, 0
    for i, p in enumerate(pairs):
        try:
            frame_img = Image.open(p["frame_path"]).convert("RGB")
            if p["qr_id"] not in ref_cache:
                ref_img = Image.open(p["ref_path"]).convert("RGB")
                gt = decode_reference_string(ref_img, p["ref_box"])
                print(f"  reference {p['qr_id']}: GT string = {gt!r}")
                if gt is None:
                    print("  WARNING: reference does not decode even at full resolution -- "
                          "decode-rate evaluation will be impossible for this QR.")
                ref_bgr = cv2.resize(cv2.cvtColor(np.array(crop_square(ref_img, p["ref_box"])), cv2.COLOR_RGB2BGR),
                                     (img_size, img_size), interpolation=cv2.INTER_AREA)
                ref_cache[p["qr_id"]] = (ref_bgr, gt)
        except Exception as e:
            print(f"  [skip] could not open pair {i}: {e}")
            n_failed += 1
            continue
        ref_bgr, gt = ref_cache[p["qr_id"]]
 
        frame_bgr = cv2.resize(cv2.cvtColor(np.array(crop_square(frame_img, p["frame_box"])), cv2.COLOR_RGB2BGR),
                               (img_size, img_size), interpolation=cv2.INTER_AREA)
        warped, score = align_reference_to_frame(frame_bgr, ref_bgr)
        scores.append(score)
        if warped is None:
            n_unaligned += 1
            continue
 
        input_path = os.path.join(cache_dir, f"{i:05d}_input.png")
        target_path = os.path.join(cache_dir, f"{i:05d}_target.png")
        cv2.imwrite(input_path, frame_bgr)
        cv2.imwrite(target_path, warped)
        entries.append({
            "input_path": input_path, "target_path": target_path,
            "qr_id": p["qr_id"], "session_id": p["session_id"], "frame_idx": p["frame_idx"],
            "aligned": True, "align_score": round(float(score), 3), "gt_string": gt,
            "blur_score": float(cv2.Laplacian(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()),
        })
 
    n_total = len(entries) + n_unaligned
    print(f"Alignment: kept {len(entries)}/{n_total} pairs (NCC >= {ALIGN_NCC_THRESH}), "
          f"dropped {n_unaligned} unaligned, {n_failed} unreadable")
    if scores:
        print(f"  alignment score percentiles (10/50/90): {np.percentile(scores, [10, 50, 90]).round(2).tolist()}")
    if n_total and len(entries) < 0.5 * n_total:
        print("  WARNING: under half the pairs aligned -- check boxes/threshold before trusting results.")
 
    # Temporal split per session: the LAST block of each video is test, with a
    # gap before it. A random split would put near-identical neighbouring
    # frames (0.1 s apart) on both sides and inflate test numbers.
    by_session = {}
    for e in entries:
        by_session.setdefault(e["session_id"], []).append(e)
    train, test = [], []
    for sid, es in by_session.items():
        es.sort(key=lambda e: e["frame_idx"])
        n_test = max(1, int(round(len(es) * test_fraction)))
        cut = len(es) - n_test
        n_gap = min(gap, max(0, cut - 1))
        train += es[:cut - n_gap]
        test += es[cut:]
 
    manifest = {"train": train, "test": test}
    json.dump(manifest, open(manifest_path, "w"))
    return manifest

In [34]:
def load_cached_image(path, expected_size):
    """Cached images are already exactly expected_size (resized once, with
    INTER_AREA). Do NOT resize again: a same-size PIL resize still runs the
    filter and degrades fine QR modules."""
    img = Image.open(path).convert("RGB")
    if img.size != (expected_size, expected_size):
        img = img.resize((expected_size, expected_size), Image.LANCZOS)
    return img

def _motion_kernel(length, angle):
    length = max(3, int(length) | 1)  # odd
    k = np.zeros((length, length), np.float32)
    k[length // 2, :] = 1.0
    M = cv2.getRotationMatrix2D(((length - 1) / 2, (length - 1) / 2), angle, 1.0)
    k = cv2.warpAffine(k, M, (length, length))
    s = k.sum()
    if s <= 0:
        k = np.zeros((length, length), np.float32)
        k[length // 2, length // 2] = 1.0
        return k
    return k / s

def synthetic_degrade(img_rgb_u8):
    """Clean uint8 RGB -> degraded uint8 RGB: motion blur, defocus,
    illumination (gamma/contrast/brightness) and sensor noise."""
    img = img_rgb_u8.astype(np.float32) / 255.0
    if random.random() < 0.85:
        k = _motion_kernel(random.randrange(3, 26), random.uniform(0, 180))
        img = cv2.filter2D(img, -1, k, borderType=cv2.BORDER_REPLICATE)
    if random.random() < 0.4:
        img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.5, 2.0))
    img = np.clip(img, 0, 1) ** random.uniform(0.7, 1.4)
    img = np.clip((img - 0.5) * random.uniform(0.7, 1.2) + 0.5 + random.uniform(-0.1, 0.1), 0, 1)
    rng = np.random.default_rng(random.getrandbits(32))
    img = img + rng.normal(0, random.uniform(0, 0.03), img.shape).astype(np.float32)
    return (np.clip(img, 0, 1) * 255).round().astype(np.uint8)

In [35]:
class QRPairDataset(Dataset):
    def __init__(self, entries, img_size, synthetic_prob=0.0):
        self.entries = entries
        self.img_size = img_size
        self.synthetic_prob = synthetic_prob
 
    def __len__(self):
        return len(self.entries)
 
    def __getitem__(self, idx):
        e = self.entries[idx]
        target = np.array(load_cached_image(e["target_path"], self.img_size))
        if self.synthetic_prob > 0 and random.random() < self.synthetic_prob:
            cond = synthetic_degrade(target)
        else:
            cond = np.array(load_cached_image(e["input_path"], self.img_size))
        cond = torch.from_numpy(cond).permute(2, 0, 1).float() / 127.5 - 1.0
        target = torch.from_numpy(target).permute(2, 0, 1).float() / 127.5 - 1.0
        return cond, target

In [36]:
def build_model(img_size=IMG_SIZE):
    """in_channels=6 because the noisy target (3ch) is concatenated with the
    degraded condition image (3ch) before being passed in. Kept deliberately
    small (block_out_channels) to stay lightweight."""
    return UNet2DModel(
        sample_size=img_size,
        in_channels=6,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(32, 64, 128),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    )

def make_schedulers(num_timesteps=TIMESTEPS):
    """Cosine schedule + v-prediction. v-prediction keeps x0 recoverable at
    every timestep (x0 = sqrt(ab)*x_t - sqrt(1-ab)*v, no division by sqrt(ab)),
    which the x0-space patch loss needs, and is well-posed at the terminal
    (near-zero SNR) step that sampling starts from ('trailing' spacing)."""
    kw = dict(num_train_timesteps=num_timesteps, beta_schedule="squaredcos_cap_v2", prediction_type="v_prediction")
    return DDPMScheduler(**kw), DDIMScheduler(**kw, timestep_spacing="trailing", clip_sample=True)
    
 
@torch.no_grad()
def ddim_sample(model, ddim_scheduler, cond, steps=SAMPLE_STEPS):
    ddim_scheduler.set_timesteps(steps, device=cond.device)
    x = torch.randn_like(cond)
    for t in ddim_scheduler.timesteps:
        v = model(torch.cat([x, cond], dim=1), t).sample
        x = ddim_scheduler.step(v, t, x).prev_sample
    return x.clamp(-1, 1)
 
 
@torch.no_grad()
def ema_update(ema_model, model, step, decay=EMA_DECAY):
    d = min(decay, (1 + step) / (10 + step))
    for pe, p in zip(ema_model.parameters(), model.parameters()):
        pe.mul_(d).add_(p.detach(), alpha=1 - d)

In [37]:
# 5. Patch-based L1 + SSIM loss 
class PatchL1SSIMLoss(torch.nn.Module):
    """Non-overlapping patch_size x patch_size tiles; per tile: L1, and
    (1 - SSIM) with the whole tile as one SSIM window, optionally plus
    log10(MSE) (== minus PSNR up to a constant). Returns a PER-SAMPLE loss [B]
    so the caller can weight by noise level. In training mode the tile grid
    is randomly offset each call so tile borders aren't fixed."""
 
    def __init__(self, patch_size=PATCH_SIZE, l1_weight=1.0, ssim_weight=1.0,
                 psnr_weight=PATCH_PSNR_WEIGHT, random_offset=RANDOM_PATCH_OFFSET):
        super().__init__()
        self.p = patch_size
        self.l1_w, self.ssim_w, self.psnr_w = l1_weight, ssim_weight, psnr_weight
        self.random_offset = random_offset
        self.c1, self.c2 = 0.01 ** 2, 0.03 ** 2
 
    def _patchify(self, x):
        p = self.p
        B, C, H, W = x.shape
        x = x[..., : H // p * p, : W // p * p]
        patches = x.unfold(2, p, p).unfold(3, p, p)            # B,C,nH,nW,p,p
        B, C, nH, nW, _, _ = patches.shape
        return patches.contiguous().view(B, C, nH * nW, p * p)  # B,C,N,p*p
 
    def forward(self, pred, target):
        pred01, target01 = (pred + 1) / 2, (target + 1) / 2
        if self.training and self.random_offset:
            dy, dx = random.randrange(self.p), random.randrange(self.p)
            pred01, target01 = pred01[..., dy:, dx:], target01[..., dy:, dx:]
        pp, tp = self._patchify(pred01), self._patchify(target01)
 
        l1 = (pp - tp).abs().mean(dim=(1, 2, 3))
 
        mu_p, mu_t = pp.mean(-1), tp.mean(-1)
        var_p, var_t = pp.var(-1, unbiased=False), tp.var(-1, unbiased=False)
        cov = ((pp - mu_p.unsqueeze(-1)) * (tp - mu_t.unsqueeze(-1))).mean(-1)
        ssim_map = ((2 * mu_p * mu_t + self.c1) * (2 * cov + self.c2)) / \
                   ((mu_p ** 2 + mu_t ** 2 + self.c1) * (var_p + var_t + self.c2))
        ssim = ssim_map.mean(dim=(1, 2))
 
        loss = self.l1_w * l1 + self.ssim_w * (1 - ssim)
        if self.psnr_w > 0:
            mse_patch = ((pp - tp) ** 2).mean(-1)
            loss = loss + self.psnr_w * torch.log10(mse_patch + 1e-4).mean(dim=(1, 2))
        return loss

In [38]:
# 6. Evaluation metrics + QR decoding
def psnr_np(pred, target, max_val=1.0):
    mse = np.mean((pred.astype(np.float64) - target.astype(np.float64)) ** 2)
    return 99.0 if mse == 0 else 10 * np.log10((max_val ** 2) / mse)


def ssim_np(pred, target):
    """Standard global windowed SSIM (Gaussian window) -- used only for reporting,
    distinct from the patch loss used during training."""
    pg = cv2.cvtColor((pred * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
    tg = cv2.cvtColor((target * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    mu1, mu2 = cv2.GaussianBlur(pg, (11, 11), 1.5), cv2.GaussianBlur(tg, (11, 11), 1.5)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = cv2.GaussianBlur(pg ** 2, (11, 11), 1.5) - mu1_sq
    s2 = cv2.GaussianBlur(tg ** 2, (11, 11), 1.5) - mu2_sq
    s12 = cv2.GaussianBlur(pg * tg, (11, 11), 1.5) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * s12 + C2)) / ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
    return float(ssim_map.mean())


def decode_qr(image_rgb_uint8):
    """Tries pyzbar first (more robust on dense QR), falls back to OpenCV."""
    try:
        from pyzbar.pyzbar import decode as zbar_decode
        results = zbar_decode(image_rgb_uint8)
        if results:
            return results[0].data.decode("utf-8", errors="ignore")
    except Exception:
        pass
    bgr = cv2.cvtColor(image_rgb_uint8, cv2.COLOR_RGB2BGR)
    data, _, _ = cv2.QRCodeDetector().detectAndDecode(bgr)
    return data if data else None

In [39]:
def find_min_decodable_size(image_path, box, sizes=(128, 160, 192, 224, 256, 320, 384, 512)):
    """Diagnostic: finds the smallest resolution this specific QR still decodes at."""
    image = Image.open(image_path).convert("RGB")
    crop = crop_square(image, box)
    crop_bgr = cv2.cvtColor(np.array(crop), cv2.COLOR_RGB2BGR)
    print(f"Original crop size: {crop.size}")
    for size in sizes:
        resized_bgr = cv2.resize(crop_bgr, (size, size), interpolation=cv2.INTER_AREA)
        resized_rgb = cv2.cvtColor(resized_bgr, cv2.COLOR_BGR2RGB)
        result = decode_qr(resized_rgb)
        print(f"  {size}x{size}: {'DECODED -> ' + result if result else 'failed'}")

In [40]:
# 7. Checkpoint save / resume via the HF dataset repo
def try_download_checkpoint(repo_id, local_path, token):
    try:
        path = hf_hub_download(repo_id=repo_id, filename="checkpoints/diffusion_qr_latest.pt",
                                repo_type="dataset", token=token)
        shutil.copy(path, local_path)
        print(f"Found existing checkpoint in {repo_id} -- resuming")
        return True
    except Exception:
        print("No existing checkpoint in the repo -- starting fresh")
        return False


def push_checkpoint(local_path, repo_id, token):
    api = HfApi(token=token)
    api.create_commit(
        repo_id=repo_id, repo_type="dataset",
        operations=[CommitOperationAdd(path_in_repo="checkpoints/diffusion_qr_latest.pt", path_or_fileobj=local_path)],
        commit_message="Update diffusion QR restoration checkpoint",
    )

In [41]:
class UNetForDP(torch.nn.Module):
    def __init__(self, unet):
        super().__init__()
        self.unet = unet
    def forward(self, x, t):
        return self.unet(x, t).sample

In [42]:
def train(model, ema_model, loader, train_scheduler, optimizer, epochs, patch_loss_fn, lambda_patch,
          start_epoch, ckpt_path, repo_id, token, save_every, fwd=None):
    fwd = fwd or (lambda x, t: model(x, t).sample)   # fwd = DataParallel-wrapped forward, or None for 1 GPU
    alphas_cumprod = train_scheduler.alphas_cumprod.to(device)
    T = train_scheduler.config.num_train_timesteps
    step = start_epoch * len(loader)
 
    for epoch in range(start_epoch, epochs):
        model.train()
        tot_v, tot_patch, n = 0.0, 0.0, 0
        for cond, x0 in loader:
            cond, x0 = cond.to(device), x0.to(device)
            b = x0.shape[0]
            t = torch.randint(0, T, (b,), device=device).long()
            noise = torch.randn_like(x0)
            x_t = train_scheduler.add_noise(x0, noise, t)
            v_target = train_scheduler.get_velocity(x0, noise, t)
 
            v_pred = fwd(torch.cat([x_t, cond], dim=1), t)
            loss_v = F.mse_loss(v_pred, v_target)
 
            # x0 estimate from v; patch loss weighted by alpha_bar so it fades
            # out at high noise levels where x0 is inherently uncertain.
            ab = alphas_cumprod[t].view(-1, 1, 1, 1)
            x0_pred = (ab.sqrt() * x_t - (1 - ab).sqrt() * v_pred).clamp(-1, 1)
            loss_patch = (patch_loss_fn(x0_pred, x0) * ab.view(-1)).mean()
 
            loss = loss_v + lambda_patch * loss_patch
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            step += 1
            ema_update(ema_model, model, step)
 
            tot_v += loss_v.item() * b
            tot_patch += loss_patch.item() * b
            n += b
 
        print(f"epoch {epoch + 1}/{epochs}  v_loss={tot_v / n:.4f}  patch_loss={tot_patch / n:.4f}")
 
        if (epoch + 1) % save_every == 0 or epoch + 1 == epochs:
            torch.save({"model": model.state_dict(), "ema": ema_model.state_dict(),
                        "optimizer": optimizer.state_dict(), "epoch": epoch + 1}, ckpt_path)
            print(f"  saved checkpoint -> {ckpt_path}")
            if repo_id:
                push_checkpoint(ckpt_path, repo_id, token)
                print(f"  pushed checkpoint -> {repo_id}/diffusion_qr_latest.pt")

In [43]:
def _to_tensor(img_u8):
    return torch.from_numpy(img_u8).permute(2, 0, 1).float().unsqueeze(0).to(device) / 127.5 - 1.0
 
 
def _to_uint8(t):
    return ((t[0].clamp(-1, 1).cpu().numpy().transpose(1, 2, 0) + 1) * 127.5).round().astype(np.uint8)
 
 
def _status(decoded, gt):
    if gt is None:
        return "no GT string"
    return "MATCH" if decoded == gt else ("wrong string" if decoded else "no decode")

In [44]:
def evaluate(sample_fn, test_entries, img_size):
    """sample_fn: cond tensor [1,3,H,W] in [-1,1] -> restored tensor.
    Decode success is judged against the GT string decoded from the original
    full-resolution reference, stratified by blur score (low Laplacian
    variance = blurrier). Also reports mean inference time per frame."""
    rows, t_total = [], 0.0
    for e in test_entries:
        cond_np = np.array(load_cached_image(e["input_path"], img_size))
        target_np = np.array(load_cached_image(e["target_path"], img_size))
        gt = e.get("gt_string")
 
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        restored = sample_fn(_to_tensor(cond_np))
        if device == "cuda":
            torch.cuda.synchronize()
        t_total += time.time() - t0
        restored_np = _to_uint8(restored)
 
        rows.append({
            "blur": e["blur_score"],
            "raw_ok": gt is not None and decode_qr(cond_np) == gt,
            "restored_ok": gt is not None and decode_qr(restored_np) == gt,
            "psnr": psnr_np(restored_np / 255.0, target_np / 255.0),
            "ssim": ssim_np(restored_np / 255.0, target_np / 255.0),
            "raw_psnr": psnr_np(cond_np / 255.0, target_np / 255.0),
            "raw_ssim": ssim_np(cond_np / 255.0, target_np / 255.0),
        })
 
    def summarize(name, rs):
        m = len(rs)
        if m == 0:
            return
        raw, res = sum(r["raw_ok"] for r in rs), sum(r["restored_ok"] for r in rs)
        print(f"  {name:<10} n={m:<3} decode: raw {raw}/{m} ({100 * raw / m:.0f}%) -> restored {res}/{m} ({100 * res / m:.0f}%)"
              f" | PSNR raw {np.mean([r['raw_psnr'] for r in rs]):.2f} -> {np.mean([r['psnr'] for r in rs]):.2f} dB"
              f" | SSIM raw {np.mean([r['raw_ssim'] for r in rs]):.3f} -> {np.mean([r['ssim'] for r in rs]):.3f}")
 
    print(f"\nEvaluation on {len(rows)} held-out frames "
          f"(decode judged vs. GT string from the full-res reference; avg {1000 * t_total / max(1, len(rows)):.0f} ms/frame):")
    summarize("ALL", rows)
    order = np.argsort([r["blur"] for r in rows])
    for name, idx in zip(("blurriest", "medium", "sharpest"), np.array_split(order, 3)):
        summarize(name, [rows[i] for i in idx])
    return rows

In [45]:
def visualize_sample(sample_fn, entry, img_size=IMG_SIZE, save_path=None):
    """visualize_sample(sample_fn, manifest["test"][0])"""
    cond_np = np.array(load_cached_image(entry["input_path"], img_size))
    target_np = np.array(load_cached_image(entry["target_path"], img_size))
    gt = entry.get("gt_string")
    restored_np = _to_uint8(sample_fn(_to_tensor(cond_np)))
    raw_s, res_s = decode_qr(cond_np), decode_qr(restored_np)
 
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
    panels = [
        (cond_np, f"Degraded input (frame)\nvs GT string: {_status(raw_s, gt)}"),
        (restored_np, f"Model output (restored)\nvs GT string: {_status(res_s, gt)}"),
        (target_np, f"Aligned reference (target)\nGT string: {'known' if gt else 'unknown'}"),
    ]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    fig.suptitle(f"qr_id={entry.get('qr_id')}  session={entry.get('session_id')}  "
                 f"align={entry.get('align_score')}  blur={entry.get('blur_score', 0):.0f}")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    return {"raw": raw_s, "restored": res_s, "gt": gt, "match": res_s is not None and res_s == gt}
 
 
def restore_and_decode(frame_path, box, sample_fn, img_size=IMG_SIZE):
    """Inference on a NEW raw frame: crop to a detected QR box (SAM3 or any
    detector) -> restore -> decode. Returns (restored_rgb, decoded_restored, decoded_raw)."""
    frame = Image.open(frame_path).convert("RGB")
    bgr = cv2.resize(cv2.cvtColor(np.array(crop_square(frame, box)), cv2.COLOR_RGB2BGR),
                     (img_size, img_size), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    restored = _to_uint8(sample_fn(_to_tensor(rgb)))
    return restored, decode_qr(restored), decode_qr(rgb)

In [46]:
@torch.no_grad()
def one_step_sample(student, train_scheduler, cond):
    T = train_scheduler.config.num_train_timesteps
    ab_T = train_scheduler.alphas_cumprod[T - 1].to(cond.device)
    z = torch.randn_like(cond)
    t = torch.full((cond.shape[0],), T - 1, device=cond.device, dtype=torch.long)
    v = student(torch.cat([z, cond], dim=1), t).sample
    return (ab_T.sqrt() * z - (1 - ab_T).sqrt() * v).clamp(-1, 1)
 
 
def distill_one_step(teacher, loader, ddim_scheduler, train_scheduler, epochs=DISTILL_EPOCHS,
                     lr=1e-4, teacher_steps=SAMPLE_STEPS):
    """Distills the multi-step teacher (DDIM, `teacher_steps`) into a ONE-step
    student of the same architecture: student(noise, cond) at t=T-1 is trained
    to reproduce the teacher's output and the ground-truth target.
    NOTE: this is plain output distillation, not FideDiff's consistency training."""
    T = train_scheduler.config.num_train_timesteps
    ab_T = train_scheduler.alphas_cumprod[T - 1].to(device)
    teacher.eval()
    student = copy.deepcopy(teacher)
    for p in student.parameters():
        p.requires_grad_(True)
    student.train()
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    patch_loss_fn = PatchL1SSIMLoss()
 
    for epoch in range(epochs):
        tot, n = 0.0, 0
        for cond, x0 in loader:
            cond, x0 = cond.to(device), x0.to(device)
            with torch.no_grad():
                target = ddim_sample(teacher, ddim_scheduler, cond, teacher_steps)
            z = torch.randn_like(cond)
            t = torch.full((cond.shape[0],), T - 1, device=device, dtype=torch.long)
            v = student(torch.cat([z, cond], dim=1), t).sample
            x0_hat = (ab_T.sqrt() * z - (1 - ab_T).sqrt() * v).clamp(-1, 1)
            loss = patch_loss_fn(x0_hat, target).mean() + 0.5 * patch_loss_fn(x0_hat, x0).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            opt.step()
            tot += loss.item() * cond.shape[0]
            n += cond.shape[0]
        print(f"distill epoch {epoch + 1}/{epochs}  loss={tot / n:.4f}")
    student.eval()
    return student

In [47]:
set_seed(SEED)
token = get_hf_token()
 
print("Pulling dataset, frames, and annotations...")
snapshot_download(
        repo_id=REPO_ID, repo_type="dataset", local_dir=LOCAL_DIR,
        allow_patterns=["raw/reference_photos/*", "frames/**/*.jpg", "annotations/**"],
        token=token,
    )

Pulling dataset, frames, and annotations...


Fetching 366 files:   0%|          | 0/366 [00:00<?, ?it/s]

'/kaggle/working/buggyqr_data'

In [48]:
manifest = prepare_training_pairs(LOCAL_DIR, CACHE_DIR, IMG_SIZE)
print(f"Train pairs: {len(manifest['train'])}   Test pairs: {len(manifest['test'])}")
if not manifest["train"] or not manifest["test"]:
    raise RuntimeError("Empty train or test split -- too few aligned pairs. Lower ALIGN_NCC_THRESH or check boxes.")
 
train_loader = DataLoader(QRPairDataset(manifest["train"], IMG_SIZE, synthetic_prob=SYNTHETIC_PROB),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
                              drop_last=len(manifest["train"]) > BATCH_SIZE)

Found existing cache -> /kaggle/working/qr_pairs_cache/manifest.json (delete the whole cache dir to rebuild)
Train pairs: 275   Test pairs: 49


In [49]:
model = build_model(IMG_SIZE).to(device)
ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad_(False)
fwd = None
if torch.cuda.device_count() > 1:
    fwd = torch.nn.DataParallel(UNetForDP(model))
    print(f"Training on {torch.cuda.device_count()} GPUs via DataParallel")
train_scheduler, ddim_scheduler = make_schedulers()
patch_loss_fn = PatchL1SSIMLoss(patch_size=PATCH_SIZE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

Training on 2 GPUs via DataParallel


In [50]:
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_path = os.path.join(CKPT_DIR, "diffusion_qr_latest.pt")
start_epoch = 0
have_ckpt = (RESUME_FROM_HUB and try_download_checkpoint(CHECKPOINT_REPO_ID, ckpt_path, token)) or os.path.exists(ckpt_path)
if have_ckpt:
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    ema_model.load_state_dict(ckpt["ema"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"]
    print(f"Resumed from checkpoint at epoch {start_epoch} (raise EPOCHS above this to keep training)")

checkpoints/diffusion_qr_latest.pt:   0%|          | 0.00/63.5M [00:00<?, ?B/s]

Found existing checkpoint in RakshitAi/IITB-BuggyQR-diffusion -- resuming
Resumed from checkpoint at epoch 40 (raise EPOCHS above this to keep training)


In [55]:
train(model, ema_model, train_loader, train_scheduler, optimizer, EPOCHS, patch_loss_fn, LAMBDA_PATCH,
          start_epoch, ckpt_path, CHECKPOINT_REPO_ID if PUSH_CHECKPOINTS else None, token, SAVE_EVERY)

epoch 41/100  v_loss=0.0781  patch_loss=0.0644
epoch 42/100  v_loss=0.0772  patch_loss=0.0521
epoch 43/100  v_loss=0.0670  patch_loss=0.0540
epoch 44/100  v_loss=0.0685  patch_loss=0.0589
epoch 45/100  v_loss=0.0799  patch_loss=0.0538
  saved checkpoint -> /kaggle/working/checkpoints/diffusion_qr_latest.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  pushed checkpoint -> RakshitAi/IITB-BuggyQR-diffusion/diffusion_qr_latest.pt
epoch 46/100  v_loss=0.0680  patch_loss=0.0566
epoch 47/100  v_loss=0.0677  patch_loss=0.0501
epoch 48/100  v_loss=0.0664  patch_loss=0.0513
epoch 49/100  v_loss=0.0614  patch_loss=0.0534
epoch 50/100  v_loss=0.0629  patch_loss=0.0515
  saved checkpoint -> /kaggle/working/checkpoints/diffusion_qr_latest.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  pushed checkpoint -> RakshitAi/IITB-BuggyQR-diffusion/diffusion_qr_latest.pt
epoch 96/100  v_loss=0.0399  patch_loss=0.0346
epoch 97/100  v_loss=0.0402  patch_loss=0.0334
epoch 98/100  v_loss=0.0424  patch_loss=0.0363
epoch 99/100  v_loss=0.0440  patch_loss=0.0359
epoch 100/100  v_loss=0.0460  patch_loss=0.0367
  saved checkpoint -> /kaggle/working/checkpoints/diffusion_qr_latest.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  pushed checkpoint -> RakshitAi/IITB-BuggyQR-diffusion/diffusion_qr_latest.pt


In [56]:
sample_fn = lambda c: ddim_sample(ema_model, ddim_scheduler, c, SAMPLE_STEPS)

In [57]:
evaluate(sample_fn, manifest["test"], IMG_SIZE)


Evaluation on 49 held-out frames (decode judged vs. GT string from the full-res reference; avg 992 ms/frame):
  ALL        n=49  decode: raw 4/49 (8%) -> restored 3/49 (6%) | PSNR raw 13.92 -> 20.99 dB | SSIM raw 0.726 -> 0.817
  blurriest  n=17  decode: raw 0/17 (0%) -> restored 0/17 (0%) | PSNR raw 12.18 -> 15.31 dB | SSIM raw 0.533 -> 0.647
  medium     n=16  decode: raw 1/16 (6%) -> restored 1/16 (6%) | PSNR raw 14.44 -> 22.28 dB | SSIM raw 0.776 -> 0.886
  sharpest   n=16  decode: raw 3/16 (19%) -> restored 2/16 (12%) | PSNR raw 15.23 -> 25.73 dB | SSIM raw 0.880 -> 0.930


[{'blur': 719.8740414222702,
  'raw_ok': False,
  'restored_ok': False,
  'psnr': np.float64(16.386511196737864),
  'ssim': 0.7078450389180908,
  'raw_psnr': np.float64(12.163433631484665),
  'raw_ssim': 0.610307707026514},
 {'blur': 157.36587396916002,
  'raw_ok': False,
  'restored_ok': False,
  'psnr': np.float64(12.775707581700615),
  'ssim': 0.4771005149230646,
  'raw_psnr': np.float64(12.30869821601794),
  'raw_ssim': 0.4538816197216087},
 {'blur': 788.8914642536547,
  'raw_ok': False,
  'restored_ok': False,
  'psnr': np.float64(20.505387002593277),
  'ssim': 0.8557931151758555,
  'raw_psnr': np.float64(16.443392194475127),
  'raw_ssim': 0.7524999669095312},
 {'blur': 215.08573866426013,
  'raw_ok': False,
  'restored_ok': False,
  'psnr': np.float64(13.694950759408364),
  'ssim': 0.5597250526925316,
  'raw_psnr': np.float64(12.842097847074411),
  'raw_ssim': 0.49070908109905587},
 {'blur': 194.27738577104174,
  'raw_ok': False,
  'restored_ok': False,
  'psnr': np.float64(12.66

In [ ]:
visualize_sample(sample_fn, manifest["test"][0])